In [ ]:
# ==========================================
# MACHINE LEARNING PIPELINE - FINAL REVISION + PREVIEW
# ==========================================

import pandas as pd
import numpy as np
import pickle
import os
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_score
from google.colab import drive

# 1. SETUP & LOAD DATA
drive.mount('/content/drive')
base_path = "/content/drive/MyDrive/Semester 7 /Computing project "
input_file = os.path.join(base_path, "user_level_features_final_for_ML.csv")

df = pd.read_csv(input_file)
print(f"Data Loaded: {df.shape[0]} students")

# 2. TRAINING
features = ['mean_score_pct', 'engagement_score', 'consistency_score', 'num_attempts']
X = df[features].copy()
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Training K-Means
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['ml_cluster'] = kmeans.fit_predict(X_scaled)
score = silhouette_score(X_scaled, df['ml_cluster'])
print(f"Silhouette Score: {score:.4f}")

# 3. INTERPRETASI CLUSTER (LABELING)
cluster_summary = df.groupby('ml_cluster')[features].mean()

def get_cluster_label(row):
    score = row['mean_score_pct']
    engage = row['engagement_score']

    if score > 70:
        return "High Performer (Star)"
    elif engage > 5000:
        if score > 50: return "Active Learner"
        else: return "Hard Worker / Struggling"
    elif engage > 1000:
        return "Balanced Learner"
    else:
        return "At Risk / Passive"

cluster_labels = {}
print("\n=== PEMETAAN CLUSTER (KAMUS) ===")
for cluster_id, row in cluster_summary.iterrows():
    label = get_cluster_label(row)
    cluster_labels[cluster_id] = label
    print(f"Cluster {cluster_id} --> {label} (Avg Score: {row['mean_score_pct']:.1f}, Avg Engage: {row['engagement_score']:.0f})")

# 4. SINKRONISASI CSV (FIX UTAMA)
df['cluster'] = df['ml_cluster'] # Update kolom cluster
df.to_csv(input_file, index=False) # Overwrite file
print(f"\n[CRITICAL FIX] File CSV berhasil di-update agar sinkron dengan model.")

# 5. FUNGSI LOGIC REKOMENDASI
def generate_recommendation(student_row, learning_style="Visual", interest="Web Development"):
    cluster_id = student_row['ml_cluster']
    cluster_type = cluster_labels.get(cluster_id, "Unknown")

    rec = {"status": cluster_type, "materials": [], "tips": ""}

    if "High Performer" in cluster_type:
        rec['materials'] = [f"Advanced {interest} Projects", "LeetCode Hard"]
        rec['tips'] = "Fokus portofolio."
    elif "Active" in cluster_type:
        rec['materials'] = [f"Studi Kasus {interest}", "Deep Dive Concepts"]
        rec['tips'] = "Pertahankan konsistensi."
    elif "Balanced" in cluster_type:
        rec['materials'] = [f"Kursus {interest} Menengah", "Latihan Soal Medium"]
        rec['tips'] = "Tingkatkan latihan soal."
    elif "Struggling" in cluster_type:
        rec['materials'] = ["Mentoring Sebaya", "Review Jawaban Salah"]
        rec['tips'] = "Review konsep dasar."
    else: # At Risk
        rec['materials'] = ["Video Ringkasan Materi (Wajib)", "Manajemen Waktu"]
        rec['tips'] = "Segera kejar ketertinggalan."

    # Personalisasi
    style = learning_style.lower()
    suffix = "(Video)" if style == "visual" else "(Podcast)" if style == "auditory" else "(Praktik)"
    rec['materials'] = [f"{m} {suffix}" for m in rec['materials']]

    return rec

# 6. SIMPAN MODEL PKL
model_data = {
    "model": kmeans, "scaler": scaler,
    "cluster_labels": cluster_labels, "features": features
}
output_model_path = os.path.join(base_path, "recommendation_engine.pkl")
with open(output_model_path, "wb") as f:
    pickle.dump(model_data, f)
print(f"[DONE] Model PKL berhasil disimpan.")

# 7. PREVIEW / SIMULASI (PROOF OF CONCEPT)
print("\n" + "="*50)
print("     VERIFIKASI HASIL DIAGNOSA")
print("="*50)

# A. Ambil 3 Sample Acak
samples = df.sample(3, random_state=1)
print("\n--- [A] 3 USER ACAK ---")
for idx, row in samples.iterrows():
    res = generate_recommendation(row, "Visual", "Data")
    print(f"User ID: {row['userid']}")
    print(f"Data Asli -> Score: {row['mean_score_pct']:.1f}, Engage: {row['engagement_score']:.0f}")
    print(f"Diagnosa  -> {res['status']}")
    print("-" * 30)

# B. Cek Khusus User Demo (8994295)
print("\n--- [B] CEK KHUSUS USER DEMO (8994295) ---")
target_user = df[df['userid'] == 8994295]

if not target_user.empty:
    row = target_user.iloc[0]
    res = generate_recommendation(row, "Auditory", "Web Dev")
    print(f"User ID   : {row['userid']}")
    print(f"Data Asli : Score {row['mean_score_pct']}, Engage {row['engagement_score']}")
    print(f"Diagnosa  : {res['status']} <--- (HARUSNYA 'At Risk / Passive')")
    print(f"Saran AI  : {res['tips']}")
else:
    print("User demo 8994295 tidak ditemukan di dataset filter ini.")

print("\n" + "="*50)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data Loaded: 561 students
Silhouette Score: 0.8363

=== PEMETAAN CLUSTER (KAMUS) ===
Cluster 0 --> At Risk / Passive (Avg Score: 0.2, Avg Engage: 253)
Cluster 1 --> Active Learner (Avg Score: 65.7, Avg Engage: 16021)
Cluster 2 --> Balanced Learner (Avg Score: 57.5, Avg Engage: 3125)
Cluster 3 --> Active Learner (Avg Score: 64.2, Avg Engage: 9749)

[CRITICAL FIX] File CSV berhasil di-update agar sinkron dengan model.
[DONE] Model PKL berhasil disimpan.

     VERIFIKASI HASIL DIAGNOSA

--- [A] 3 USER ACAK ---
User ID: 23450
Data Asli -> Score: 0.0, Engage: 9
Diagnosa  -> At Risk / Passive
------------------------------
User ID: 6307583
Data Asli -> Score: 0.0, Engage: 134
Diagnosa  -> At Risk / Passive
------------------------------
User ID: 104531
Data Asli -> Score: 63.7, Engage: 12936
Diagnosa  -> Active Learner
------------------------------

--- [B] CEK KH

In [ ]:
# ==============================================================================
# CELL 1: DATA ENGINEERING (MULTI-FILE UPLOAD, MERGE & CLEANING)
# ==============================================================================

from google.colab import files
import pandas as pd
import numpy as np
import os
import io
import re
import html

# 1. BERSIHKAN LINGKUNGAN LAMA
!rm *.xlsx *.csv 2>/dev/null

print("="*60)
print("📥 SILAKAN UPLOAD SEMUA FILE RAW (LOGS & NILAI) SEKALIGUS")
print("   (Total ada sekitar 12 file berdasarkan screenshot Anda)")
print("="*60)

uploaded = files.upload()

# 2. PISAHKAN FILE SCORE DAN LOGS OTOMATIS
score_files = []
log_files = []

print("\n📦 Mengkategorikan File...")
for filename in uploaded.keys():
    # Jika nama file mengandung 'logs_', masuk ke logs
    if "logs_" in filename.lower():
        log_files.append(filename)
        print(f"   📂 [LOGS]  {filename}")
    else:
        # Sisanya diasumsikan file nilai (Logika/Matdis)
        score_files.append(filename)
        print(f"   📊 [SCORE] {filename}")

# 3. PROSES PENGGABUNGAN (MERGING)
print("\n🔄 Menggabungkan Dataframe...")

# A. Gabung Score
dfs_score = []
for fn in score_files:
    try:
        # Baca semua sebagai string agar aman
        df = pd.read_excel(io.BytesIO(uploaded[fn]), dtype=str, engine='openpyxl')
        dfs_score.append(df)
    except Exception as e:
        print(f"❌ Gagal baca {fn}: {e}")

if dfs_score:
    score_df = pd.concat(dfs_score, ignore_index=True)
    print(f"✅ Total Data Nilai: {len(score_df)} baris")
else:
    raise ValueError("Tidak ada file nilai yang terupload!")

# B. Gabung Logs
dfs_logs = []
for fn in log_files:
    try:
        df = pd.read_excel(io.BytesIO(uploaded[fn]))
        dfs_logs.append(df)
    except Exception as e:
        print(f"❌ Gagal baca {fn}: {e}")

if dfs_logs:
    logs_df = pd.concat(dfs_logs, ignore_index=True)
    print(f"✅ Total Data Logs: {len(logs_df)} baris")
else:
    print("⚠️ Warning: Tidak ada file logs. Membuat dummy.")
    logs_df = pd.DataFrame(columns=['Time', 'Description'])

# 4. DATA CLEANING (SAPU JAGAT)
print("\n🧹 Memulai Pembersihan Data...")

def clean_indo_number(val):
    val_str = str(val).strip().lower()
    if val_str in ['nan', 'none', '', 'null', 'nat', '-']: return np.nan
    val_str = val_str.replace(',', '.')
    try:
        f = float(val_str)
        if f < 0: return 0
        if f > 100: return 100
        return f
    except:
        return np.nan

# Strategi Jala Ikan (Cari nilai di berbagai kemungkinan kolom)
cols_to_check = ['final_quiz_grade', 'assignmentscore', 'raw_quiz_score', 'Grade']
score_df['final_score_fixed'] = np.nan

for col in cols_to_check:
    if col in score_df.columns:
        print(f"   👉 Membersihkan kolom: {col}")
        cleaned_col = score_df[col].apply(clean_indo_number)
        # Isi yang kosong dengan nilai dari kolom ini
        score_df['final_score_fixed'] = score_df['final_score_fixed'].fillna(cleaned_col)

# Update kolom utama
score_df['final_quiz_grade'] = score_df['final_score_fixed']

# Fix Nama Aktivitas
name_cols = ['quizname', 'assignmentname', 'Event context']
score_df['final_activity_name'] = np.nan
for col in name_cols:
    if col in score_df.columns:
        score_df['final_activity_name'] = score_df['final_activity_name'].fillna(score_df[col])
score_df['quizname'] = score_df['final_activity_name'].fillna("Unknown Activity")

# Fix User ID
score_df['userid'] = pd.to_numeric(score_df['userid'], errors='coerce').fillna(0).astype(int)
score_df = score_df[score_df['userid'] > 0]

# 5. VERIFIKASI KHUSUS (PROOF OF CONCEPT)
print("\n🔍 [VERIFIKASI DATA KRUSIAL]")

# Cek User 66745 (Kasus Nilai Hilang)
u_check = 66745
cek_df = score_df[score_df['userid'] == u_check]
if not cek_df.empty:
    # Cari nilai rata-rata
    avg_val = cek_df['final_quiz_grade'].mean()
    print(f"   👤 User {u_check}:")
    print(f"      - Jumlah Aktivitas: {len(cek_df)}")
    print(f"      - Rata-rata Nilai: {avg_val:.2f}")
    if avg_val > 0:
        print("      ✅ STATUS: AMAN. Nilai terbaca.")
    else:
        print("      ⚠️ STATUS: MASIH 0. Cek file raw apakah nilainya ada?")

    # Tampilkan sampel
    print("      - Sampel Data:")
    print(cek_df[['quizname', 'final_quiz_grade']].head(3).to_string(index=False))
else:
    print(f"   ⚠️ User {u_check} tidak ditemukan di file yang diupload.")

# 6. AGGREGASI & LOGS PROCESSING
print("\n⚙️ Memproses Agregasi & Logs...")

# Agregasi Score
score_agg = score_df.groupby("userid").agg(
    num_attempts = ("quiz_state", lambda x: (x.astype(str) == "finished").sum()) if 'quiz_state' in score_df.columns else ("userid", "count"),
    mean_score_pct = ("final_quiz_grade", "mean"),
    num_quizzes_taken = ("userid", "count")
).reset_index()

# Logs Cleaning
logs_df["Time_parsed"] = pd.to_datetime(logs_df["Time"], dayfirst=True, errors='coerce')
def extract_userid(txt):
    if pd.isna(txt): return np.nan
    t = html.unescape(str(txt))
    m = re.search(r"id\s+'?(\d+)'?", t)
    return int(m.group(1)) if m else np.nan

logs_df['actor_userid'] = logs_df['Description'].apply(extract_userid)
logs_clean = logs_df.dropna(subset=['actor_userid']).copy()
logs_clean['actor_userid'] = logs_clean['actor_userid'].astype(int)

logs_agg = logs_clean.groupby("actor_userid").agg(
    total_events = ("Time_parsed", "size"),
    last_activity = ("Time_parsed", "max"),
    first_activity = ("Time_parsed", "min")
).reset_index().rename(columns={"actor_userid": "userid"})

logs_agg['active_days'] = (logs_agg['last_activity'] - logs_agg['first_activity']).dt.days
logs_agg['active_days'] = logs_agg['active_days'].replace(0, 1)
logs_agg['events_per_day'] = logs_agg['total_events'] / logs_agg['active_days']

# 7. MERGE FINAL & SAVE
print("🛠️ Final Merge...")
user_level_df = pd.merge(score_agg, logs_agg, on="userid", how="outer").fillna(0)
user_level_df['mean_score_pct'] = user_level_df['mean_score_pct'].round(2)
user_level_df['engagement_score'] = user_level_df['total_events']
user_level_df['consistency_score'] = user_level_df['events_per_day']

# SIMPAN KE FOLDER DRIVE
from google.colab import drive
drive.mount('/content/drive')
save_path = "/content/drive/MyDrive/Semester 7 /Computing project /FINAL_RAW_MERGED_V2"
if not os.path.exists(save_path): os.makedirs(save_path)

score_df.to_csv(f"{save_path}/merged_score_data_cleaned.csv", index=False)
user_level_df.to_csv(f"{save_path}/user_level_features_final_for_ML.csv", index=False)

print("\n" + "="*50)
print("✅ DE SELESAI. File tersimpan di Google Drive.")
print(f"📂 {save_path}")
print("="*50)

📥 SILAKAN UPLOAD SEMUA FILE RAW (LOGS & NILAI) SEKALIGUS
   (Total ada sekitar 12 file berdasarkan screenshot Anda)


Saving LOGIKA MATEMATIKA IF-48-01PJJ [IZA].xlsx to LOGIKA MATEMATIKA IF-48-01PJJ [IZA].xlsx
Saving LOGIKA MATEMATIKA IF-48-02PJJ [IZA].xlsx to LOGIKA MATEMATIKA IF-48-02PJJ [IZA].xlsx
Saving LOGIKA MATEMATIKA IF-48-03PJJ [LZD].xlsx to LOGIKA MATEMATIKA IF-48-03PJJ [LZD].xlsx
Saving MATEMATIKA DISKRIT IF-48-01PJJ [DTO].xlsx to MATEMATIKA DISKRIT IF-48-01PJJ [DTO].xlsx
Saving MATEMATIKA DISKRIT IF-48-02PJJ [DTO].xlsx to MATEMATIKA DISKRIT IF-48-02PJJ [DTO].xlsx
Saving MATEMATIKA DISKRIT IF-48-03PJJ [FTY].xlsx to MATEMATIKA DISKRIT IF-48-03PJJ [FTY].xlsx
Saving logs_CAK1DAB3-IF-48-01PJJ_20251113-1315-1419.xlsx to logs_CAK1DAB3-IF-48-01PJJ_20251113-1315-1419.xlsx
Saving logs_CAK1DAB3-IF-48-01PJJ_20251113-1354-1942.xlsx to logs_CAK1DAB3-IF-48-01PJJ_20251113-1354-1942.xlsx
Saving logs_CAK1DAB3-IF-48-03PJJ_20251113-1123-2136.xlsx to logs_CAK1DAB3-IF-48-03PJJ_20251113-1123-2136.xlsx
Saving logs_CAK1EAB3-IF-48-01PJJ_20251113-1049-1948.xlsx to logs_CAK1EAB3-IF-48-01PJJ_20251113-1049-1948.xlsx
Sa

/tmp/ipython-input-915951321.py:148: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  logs_df["Time_parsed"] = pd.to_datetime(logs_df["Time"], dayfirst=True, errors='coerce')


🛠️ Final Merge...
Mounted at /content/drive

✅ DE SELESAI. File tersimpan di Google Drive.
📂 /content/drive/MyDrive/Semester 7 /Computing project /FINAL_RAW_MERGED_V2


In [ ]:
# ==============================================================================
# CELL 2: MACHINE LEARNING (FINAL MODELING)
# ==============================================================================
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_score

# 1. LOAD FROM NEW FOLDER
save_path = "/content/drive/MyDrive/Semester 7 /Computing project /FINAL_RAW_MERGED_V2"
input_file = os.path.join(save_path, "user_level_features_final_for_ML.csv")

print(f"🔄 Memuat data ML dari: {input_file}")
df = pd.read_csv(input_file)
print(f"✅ Data Loaded: {len(df)} Students")

# 2. TRAINING
features = ['mean_score_pct', 'engagement_score', 'consistency_score', 'num_attempts']
X = df[features].copy()
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['ml_cluster'] = kmeans.fit_predict(X_scaled)

print(f"📊 Silhouette Score: {silhouette_score(X_scaled, df['ml_cluster']):.4f}")

# 3. LABELING
cluster_summary = df.groupby('ml_cluster')[features].mean()

def get_cluster_label(row):
    score = row['mean_score_pct']
    engage = row['engagement_score']

    if score > 70: return "High Performer (Star)"
    elif engage > 5000:
        if score > 50: return "Active Learner"
        else: return "Hard Worker / Struggling"
    elif engage > 1000: return "Balanced Learner"
    else: return "At Risk / Passive"

cluster_labels = {}
print("\n=== CLUSTER DEFINITION ===")
for cluster_id, row in cluster_summary.iterrows():
    label = get_cluster_label(row)
    cluster_labels[cluster_id] = label
    print(f"Cluster {cluster_id} -> {label} (Score: {row['mean_score_pct']:.1f}, Engage: {row['engagement_score']:.0f})")

# 4. SYNC CSV & SAVE MODEL
df['cluster'] = df['ml_cluster']
df.to_csv(input_file, index=False)

model_data = {
    "model": kmeans, "scaler": scaler,
    "cluster_labels": cluster_labels, "features": features
}
pkl_path = os.path.join(save_path, "recommendation_engine.pkl")
with open(pkl_path, "wb") as f:
    pickle.dump(model_data, f)

print("\n" + "="*50)
print("✅ PROSES SELESAI!")
print("📥 Silakan Download 3 File ini dari Google Drive dan pasang di Backend:")
print(f"   1. {save_path}/merged_score_data_cleaned.csv")
print(f"   2. {save_path}/user_level_features_final_for_ML.csv")
print(f"   3. {save_path}/recommendation_engine.pkl")
print("="*50)

🔄 Memuat data ML dari: /content/drive/MyDrive/Semester 7 /Computing project /FINAL_RAW_MERGED_V2/user_level_features_final_for_ML.csv
✅ Data Loaded: 444 Students
📊 Silhouette Score: 0.7847

=== CLUSTER DEFINITION ===
Cluster 0 -> At Risk / Passive (Score: 0.5, Engage: 151)
Cluster 1 -> Hard Worker / Struggling (Score: 5.6, Engage: 16372)
Cluster 2 -> Active Learner (Score: 68.6, Engage: 7542)
Cluster 3 -> Balanced Learner (Score: 59.2, Engage: 2042)

✅ PROSES SELESAI!
📥 Silakan Download 3 File ini dari Google Drive dan pasang di Backend:
   1. /content/drive/MyDrive/Semester 7 /Computing project /FINAL_RAW_MERGED_V2/merged_score_data_cleaned.csv
   2. /content/drive/MyDrive/Semester 7 /Computing project /FINAL_RAW_MERGED_V2/user_level_features_final_for_ML.csv
   3. /content/drive/MyDrive/Semester 7 /Computing project /FINAL_RAW_MERGED_V2/recommendation_engine.pkl


In [ ]:
# ==============================================================================
# CELL 1: DATA ENGINEERING FINAL (FIX IMPORT ERROR)
# ==============================================================================

from google.colab import files
import pandas as pd
import numpy as np
import os
import io
import re
import html # Pastikan ini ada

# Cek apakah file sudah diupload? Jika belum, upload dulu.
if not os.path.exists("merged_score_data_cleaned.csv"): # Cek dummy file
    print("📂 Menggunakan file yang sudah ada di memori...")
else:
    print("⚠️ File lama ditemukan, tapi kita proses ulang dari raw.")

# ---------------------------------------------------------
# LANGKAH 1: IDENTIFIKASI & PENGGABUNGAN FILE
# ---------------------------------------------------------
score_frames = []
log_frames = []

# Kita cari file excel di folder saat ini
all_files = [f for f in os.listdir('.') if f.endswith('.xlsx')]

print(f"\n📦 [PROCESS 1] Mengidentifikasi {len(all_files)} File Excel...")

for fn in all_files:
    try:
        if "logs_" in fn.lower():
            print(f"   📂 LOGS: {fn}")
            df = pd.read_excel(fn)
            log_frames.append(df)
        elif "merged_" not in fn.lower(): # Hindari file merged lama
            print(f"   📊 SCORE: {fn}")
            # Force String
            df = pd.read_excel(fn, dtype=str, engine='openpyxl')
            score_frames.append(df)
    except Exception as e:
        print(f"   ❌ Gagal baca {fn}: {e}")

if not score_frames:
    print("❌ Tidak ada file skor ditemukan! Upload ulang file excel raw.")
    # Fallback upload jika kosong
    uploaded = files.upload()
    # (Copy paste logika upload di sini jika mau, tapi asumsi file sudah ada)

# Gabungkan
score_df = pd.concat(score_frames, ignore_index=True)
logs_df = pd.concat(log_frames, ignore_index=True) if log_frames else pd.DataFrame()

print(f"\n✅ Data Tergabung: {len(score_df)} Baris Nilai, {len(logs_df)} Baris Logs.")

# ---------------------------------------------------------
# LANGKAH 2: DATA CLEANING
# ---------------------------------------------------------
print("\n🧹 [PROCESS 2] Cleaning Angka...")

def clean_indo_number(val):
    val_str = str(val).strip().lower()
    if val_str in ['nan', 'none', '', 'null', 'nat', '-']: return np.nan
    val_str = val_str.replace(',', '.')
    try:
        f = float(val_str)
        if f < 0: return 0
        if f > 100: return 100
        return f
    except:
        return np.nan

# Strategi Sapu Jagat
score_df['grade_quiz'] = score_df['final_quiz_grade'].apply(clean_indo_number) if 'final_quiz_grade' in score_df.columns else np.nan

if 'assignmentscore' in score_df.columns:
    score_df['grade_assign'] = score_df['assignmentscore'].apply(clean_indo_number)
else:
    score_df['grade_assign'] = np.nan

if 'raw_quiz_score' in score_df.columns:
    score_df['grade_raw'] = score_df['raw_quiz_score'].apply(clean_indo_number)
else:
    score_df['grade_raw'] = np.nan

score_df['final_score_fixed'] = score_df['grade_quiz'].fillna(score_df['grade_assign']).fillna(score_df['grade_raw'])
score_df['final_quiz_grade'] = score_df['final_score_fixed']

# Fix User ID
score_df['userid'] = pd.to_numeric(score_df['userid'], errors='coerce').fillna(0).astype(int)
score_df = score_df[score_df['userid'] > 0]

# Fix Nama
if 'assignmentname' not in score_df.columns: score_df['assignmentname'] = np.nan
if 'quizname' not in score_df.columns: score_df['quizname'] = np.nan
score_df['quizname'] = score_df['quizname'].fillna(score_df['assignmentname']).fillna("Unknown Activity")

# ---------------------------------------------------------
# LANGKAH 3: AGGREGASI DATA
# ---------------------------------------------------------
print("\n⚙️ [PROCESS 3] Menghitung Statistik...")

score_agg = score_df.groupby("userid").agg(
    mean_score_pct = ("final_quiz_grade", "mean"),
    num_quizzes_taken = ("userid", "count"),
    num_attempts = ("quiz_state", lambda x: (x.astype(str) == "finished").sum()) if 'quiz_state' in score_df.columns else ("userid", "count")
).reset_index()

score_agg['mean_score_pct'] = score_agg['mean_score_pct'].round(2)

# ---------------------------------------------------------
# LANGKAH 4: PROSES LOGS (ENGAGEMENT)
# ---------------------------------------------------------
print("   Memproses Logs...")
logs_df["Time_parsed"] = pd.to_datetime(logs_df["Time"], dayfirst=True, errors='coerce')

def extract_userid(txt):
    if pd.isna(txt): return np.nan
    t = html.unescape(str(txt))
    import re # Import Regex di dalam fungsi biar aman
    m = re.search(r"id\s+'?(\d+)'?", t)
    return int(m.group(1)) if m else np.nan

logs_df['actor_userid'] = logs_df['Description'].apply(extract_userid)
logs_clean = logs_df.dropna(subset=['actor_userid'])
logs_clean['actor_userid'] = logs_clean['actor_userid'].astype(int)

logs_agg = logs_clean.groupby("actor_userid").agg(
    engagement_score = ("Time_parsed", "size"),
    last_activity = ("Time_parsed", "max"),
    first_activity = ("Time_parsed", "min")
).reset_index().rename(columns={"actor_userid": "userid"})

logs_agg['active_days'] = (logs_agg['last_activity'] - logs_agg['first_activity']).dt.days
logs_agg['active_days'] = logs_agg['active_days'].replace(0, 1)
logs_agg['consistency_score'] = (logs_agg['engagement_score'] / logs_agg['active_days']).round(2)

# ---------------------------------------------------------
# LANGKAH 5: MERGE & SAVE
# ---------------------------------------------------------
print("🛠️ [PROCESS 4] Final Merge...")

final_df = pd.merge(score_agg, logs_agg, on="userid", how="outer").fillna(0)

def get_category(score):
    if score >= 80: return "High"
    elif score >= 60: return "Medium"
    else: return "Low"

final_df['performance_category'] = final_df['mean_score_pct'].apply(get_category)

# Simpan
from google.colab import drive
drive.mount('/content/drive')
save_path = "/content/drive/MyDrive/Semester 7 /Computing project /FINAL_FIX_V7_before"
if not os.path.exists(save_path): os.makedirs(save_path)

score_df.to_csv(f"{save_path}/merged_score_data_cleaned.csv", index=False)
final_df.to_csv(f"{save_path}/user_level_features_final_for_ML.csv", index=False)

print("\n" + "="*50)
print("✅ DATA ENGINEERING SUKSES! (ERROR FIXED)")
print(f"📂 File tersimpan di: {save_path}")
print("="*50)

📂 Menggunakan file yang sudah ada di memori...

📦 [PROCESS 1] Mengidentifikasi 12 File Excel...
   📊 SCORE: LOGIKA MATEMATIKA IF-48-01PJJ [IZA].xlsx
   📊 SCORE: MATEMATIKA DISKRIT IF-48-02PJJ [DTO].xlsx
   📂 LOGS: logs_CAK1DAB3-IF-48-01PJJ_20251113-1354-1942.xlsx
   📂 LOGS: logs_CAK1DAB3-IF-48-03PJJ_20251113-1123-2136.xlsx
   📂 LOGS: logs_CAK1DAB3-IF-48-01PJJ_20251113-1315-1419.xlsx
   📊 SCORE: MATEMATIKA DISKRIT IF-48-03PJJ [FTY].xlsx
   📊 SCORE: LOGIKA MATEMATIKA IF-48-03PJJ [LZD].xlsx
   📂 LOGS: logs_CAK1EAB3-IF-48-02PJJ_20251113-1008-1572.xlsx
   📊 SCORE: MATEMATIKA DISKRIT IF-48-01PJJ [DTO].xlsx
   📂 LOGS: logs_CAK1EAB3-IF-48-03PJJ_20251113-0921-1421.xlsx
   📂 LOGS: logs_CAK1EAB3-IF-48-01PJJ_20251113-1049-1948.xlsx
   📊 SCORE: LOGIKA MATEMATIKA IF-48-02PJJ [IZA].xlsx

✅ Data Tergabung: 9570 Baris Nilai, 723958 Baris Logs.

🧹 [PROCESS 2] Cleaning Angka...

⚙️ [PROCESS 3] Menghitung Statistik...
   Memproses Logs...


/tmp/ipython-input-1286223905.py:115: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  logs_df["Time_parsed"] = pd.to_datetime(logs_df["Time"], dayfirst=True, errors='coerce')
/tmp/ipython-input-1286223905.py:126: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  logs_clean['actor_userid'] = logs_clean['actor_userid'].astype(int)


🛠️ [PROCESS 4] Final Merge...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✅ DATA ENGINEERING SUKSES! (ERROR FIXED)
📂 File tersimpan di: /content/drive/MyDrive/Semester 7 /Computing project /FINAL_FIX_V7_before


In [ ]:
# ==============================================================================
# CELL 2: MACHINE LEARNING PIPELINE (CLUSTERING & LABELING)
# ==============================================================================
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_score

# 1. LOAD DATA
save_path = "/content/drive/MyDrive/Semester 7 /Computing project /FINAL_FIX_V7"
input_file = os.path.join(save_path, "user_level_features_final_for_ML.csv")

print(f"🔄 Memuat data ML dari: {input_file}")
if not os.path.exists(input_file):
    print("❌ File tidak ditemukan! Jalankan DE dulu.")
else:
    df = pd.read_csv(input_file)
    print(f"✅ Data Loaded: {len(df)} Students")

    # 2. TRAINING K-MEANS
    # Menggunakan fitur: Nilai, Engagement, Konsistensi
    features = ['mean_score_pct', 'engagement_score', 'consistency_score', 'num_attempts']
    X = df[features].copy()

    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
    df['ml_cluster'] = kmeans.fit_predict(X_scaled)

    print(f"📊 Silhouette Score: {silhouette_score(X_scaled, df['ml_cluster']):.4f}")

    # 3. LABELING (MEMBERI NAMA CLUSTER)
    # Ini menjawab pertanyaan Anda: "Siapa yang nentuin label?"
    # Jawab: KITA (Manusia) yang menentukan berdasarkan rata-rata statistik.

    cluster_summary = df.groupby('ml_cluster')[features].mean()

    def get_cluster_label(row):
        score = row['mean_score_pct']
        engage = row['engagement_score']

        # Logika Bisnis Kita:
        if score > 70: return "High Performer (Star)"
        elif engage > 5000:
            if score > 50: return "Active Learner"
            else: return "Hard Worker / Struggling"
        elif engage > 1000: return "Balanced Learner"
        else: return "At Risk / Passive"

    cluster_labels = {}
    print("\n=== CLUSTER INTERPRETATION ===")
    for cluster_id, row in cluster_summary.iterrows():
        label = get_cluster_label(row)
        cluster_labels[cluster_id] = label
        print(f"Cluster {cluster_id} -> {label} (Avg Score: {row['mean_score_pct']:.1f}, Avg Engage: {row['engagement_score']:.0f})")

    # 4. SAVE & SYNC
    # Simpan kembali Cluster ID ke CSV agar Backend tidak bingung
    df['cluster'] = df['ml_cluster']
    df.to_csv(input_file, index=False)

    model_data = {
        "model": kmeans, "scaler": scaler,
        "cluster_labels": cluster_labels, "features": features
    }
    pkl_path = os.path.join(save_path, "recommendation_engine.pkl")
    with open(pkl_path, "wb") as f:
        pickle.dump(model_data, f)

    print(f"\n✅ ML SELESAI. Silakan download 3 file dari '{save_path}'")

🔄 Memuat data ML dari: /content/drive/MyDrive/Semester 7 /Computing project /FINAL_FIX_V7/user_level_features_final_for_ML.csv
✅ Data Loaded: 444 Students
📊 Silhouette Score: 0.7847

=== CLUSTER INTERPRETATION ===
Cluster 0 -> At Risk / Passive (Avg Score: 0.5, Avg Engage: 151)
Cluster 1 -> Hard Worker / Struggling (Avg Score: 5.6, Avg Engage: 16372)
Cluster 2 -> Active Learner (Avg Score: 68.6, Avg Engage: 7542)
Cluster 3 -> Balanced Learner (Avg Score: 59.2, Avg Engage: 2042)

✅ ML SELESAI. Silakan download 3 file dari '/content/drive/MyDrive/Semester 7 /Computing project /FINAL_FIX_V7'


In [ ]:
# ==============================================================================
# CELL 1: DATA ENGINEERING FINAL (FIXED UPLOAD FLOW)
# ==============================================================================

from google.colab import files
import pandas as pd
import numpy as np
import os
import io
import re
import html

# 1. BERSIHKAN LINGKUNGAN
print("🧹 Membersihkan file lama...")
!rm *.xlsx *.csv 2>/dev/null

# 2. UPLOAD FILE (LANGKAH PERTAMA & WAJIB)
print("="*60)
print("📥 SILAKAN UPLOAD SEMUA FILE RAW (LOGS & NILAI) SEKALIGUS")
print("   (Total 12 File: 6 File Nilai Matkul + 6 File Logs)")
print("="*60)

uploaded = files.upload()

# 3. PROSES FILE (SETELAH UPLOAD)
score_frames = []
log_frames = []

print("\n📦 [PROCESS 1] Mengkategorikan & Membaca File...")

for fn in uploaded.keys():
    try:
        # Deteksi Jenis File berdasarkan Nama
        if "logs_" in fn.lower():
            print(f"   📂 LOGS: {fn}")
            df = pd.read_excel(io.BytesIO(uploaded[fn]))
            log_frames.append(df)
        else:
            print(f"   📊 SCORE: {fn}")
            # Force String untuk amankan koma
            df = pd.read_excel(io.BytesIO(uploaded[fn]), dtype=str, engine='openpyxl')
            score_frames.append(df)
    except Exception as e:
        print(f"   ❌ Gagal baca {fn}: {e}")

# Validasi
if not score_frames:
    raise ValueError("❌ ERROR: Tidak ada file skor/nilai yang terdeteksi! Cek nama file Anda.")

# 4. PENGGABUNGAN (CONCATENATE)
print("\n🔄 [PROCESS 2] Menggabungkan Data...")
score_df = pd.concat(score_frames, ignore_index=True)
logs_df = pd.concat(log_frames, ignore_index=True) if log_frames else pd.DataFrame()

print(f"✅ Data Tergabung: {len(score_df)} Baris Nilai, {len(logs_df)} Baris Logs.")

# 5. DATA CLEANING (SAPU JAGAT)
print("\n🧹 [PROCESS 3] Membersihkan Data Angka...")

def clean_indo_number(val):
    val_str = str(val).strip().lower()
    if val_str in ['nan', 'none', '', 'null', 'nat', '-']: return np.nan
    val_str = val_str.replace(',', '.')
    try:
        f = float(val_str)
        if f < 0: return 0
        if f > 100: return 100
        return f
    except: return np.nan

# Strategi Gabungan Kolom
score_df['grade_quiz'] = score_df['final_quiz_grade'].apply(clean_indo_number) if 'final_quiz_grade' in score_df.columns else np.nan

if 'assignmentscore' in score_df.columns:
    score_df['grade_assign'] = score_df['assignmentscore'].apply(clean_indo_number)
else:
    score_df['grade_assign'] = np.nan

if 'raw_quiz_score' in score_df.columns:
    score_df['grade_raw'] = score_df['raw_quiz_score'].apply(clean_indo_number)
else:
    score_df['grade_raw'] = np.nan

# Coalesce (Prioritas)
score_df['final_score_fixed'] = score_df['grade_quiz'].fillna(score_df['grade_assign']).fillna(score_df['grade_raw'])
score_df['final_quiz_grade'] = score_df['final_score_fixed']

# Fix ID & Name
score_df['userid'] = pd.to_numeric(score_df['userid'], errors='coerce').fillna(0).astype(int)
score_df = score_df[score_df['userid'] > 0]

if 'assignmentname' not in score_df.columns: score_df['assignmentname'] = np.nan
if 'quizname' not in score_df.columns: score_df['quizname'] = np.nan
score_df['quizname'] = score_df['quizname'].fillna(score_df['assignmentname']).fillna("Unknown Activity")

# 6. VERIFIKASI DATA (User 66745)
print("\n🔍 [VERIFIKASI DATA KRUSIAL]")
u_check = 66745
cek_df = score_df[score_df['userid'] == u_check]
if not cek_df.empty:
    avg_val = cek_df['final_quiz_grade'].mean()
    print(f"   👤 User {u_check} (Matdis/Logmat):")
    print(f"      - Rata-rata Nilai: {avg_val:.2f}")
    if avg_val > 0: print("      ✅ STATUS: AMAN. Data nilai terbaca.")
    else: print("      ⚠️ STATUS: MASIH 0. Cek file raw.")
else:
    print(f"   ⚠️ User {u_check} tidak ditemukan.")

# 7. AGGREGASI & LOGS
print("\n⚙️ [PROCESS 4] Agregasi & Logs Processing...")

score_agg = score_df.groupby("userid").agg(
    num_attempts = ("quiz_state", lambda x: (x.astype(str) == "finished").sum()) if 'quiz_state' in score_df.columns else ("userid", "count"),
    mean_score_pct = ("final_quiz_grade", "mean"),
    num_quizzes_taken = ("userid", "count")
).reset_index()
score_agg['mean_score_pct'] = score_agg['mean_score_pct'].round(2)

# Logs
logs_df["Time_parsed"] = pd.to_datetime(logs_df["Time"], dayfirst=True, errors='coerce')
def extract_userid(txt):
    if pd.isna(txt): return np.nan
    t = html.unescape(str(txt))
    m = re.search(r"id\s+'?(\d+)'?", t)
    return int(m.group(1)) if m else np.nan

logs_df['actor_userid'] = logs_df['Description'].apply(extract_userid)
logs_clean = logs_df.dropna(subset=['actor_userid']).copy()
logs_clean['actor_userid'] = logs_clean['actor_userid'].astype(int)

logs_agg = logs_clean.groupby("actor_userid").agg(
    engagement_score = ("Time_parsed", "size"),
    last_activity = ("Time_parsed", "max"),
    first_activity = ("Time_parsed", "min")
).reset_index().rename(columns={"actor_userid": "userid"})

logs_agg['active_days'] = (logs_agg['last_activity'] - logs_agg['first_activity']).dt.days
logs_agg['active_days'] = logs_agg['active_days'].replace(0, 1)
logs_agg['consistency_score'] = (logs_agg['engagement_score'] / logs_agg['active_days']).round(2)

# 8. MERGE & SAVE
print("🛠️ Final Merge...")
final_df = pd.merge(score_agg, logs_agg, on="userid", how="outer").fillna(0)

def get_category(score):
    if score >= 80: return "High"
    elif score >= 60: return "Medium"
    else: return "Low"
final_df['performance_category'] = final_df['mean_score_pct'].apply(get_category)

# SIMPAN KE GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')
save_path = "/content/drive/MyDrive/Semester 7 /Computing project /FINAL_FIX_V7"
if not os.path.exists(save_path): os.makedirs(save_path)

print("\n💾 Menyimpan 4 File Wajib...")
# 1. Score Cleaned (Untuk Grafik & Nilai)
score_df.to_csv(f"{save_path}/merged_score_data_cleaned.csv", index=False)
# 2. User Features (Untuk ML & Dashboard)
final_df.to_csv(f"{save_path}/user_level_features_final_for_ML.csv", index=False)
# 3. Logs Cleaned (Untuk Jadwal Belajar Optimal - WAJIB ADA)
logs_clean.to_csv(f"{save_path}/merged_logs_data_cleaned.csv", index=False)

print("\n" + "="*50)
print("✅ DATA ENGINEERING SELESAI & SUKSES!")
print(f"📂 Folder Output: {save_path}")
print("   (File ke-4 'recommendation_engine.pkl' akan dibuat di tahap ML)")
print("="*50)

🧹 Membersihkan file lama...
📥 SILAKAN UPLOAD SEMUA FILE RAW (LOGS & NILAI) SEKALIGUS
   (Total 12 File: 6 File Nilai Matkul + 6 File Logs)


Saving LOGIKA MATEMATIKA IF-48-01PJJ [IZA].xlsx to LOGIKA MATEMATIKA IF-48-01PJJ [IZA].xlsx
Saving LOGIKA MATEMATIKA IF-48-02PJJ [IZA].xlsx to LOGIKA MATEMATIKA IF-48-02PJJ [IZA].xlsx
Saving LOGIKA MATEMATIKA IF-48-03PJJ [LZD].xlsx to LOGIKA MATEMATIKA IF-48-03PJJ [LZD].xlsx
Saving logs_CAK1DAB3-IF-48-01PJJ_20251113-1315-1419.xlsx to logs_CAK1DAB3-IF-48-01PJJ_20251113-1315-1419.xlsx
Saving logs_CAK1DAB3-IF-48-01PJJ_20251113-1354-1942.xlsx to logs_CAK1DAB3-IF-48-01PJJ_20251113-1354-1942.xlsx
Saving logs_CAK1DAB3-IF-48-03PJJ_20251113-1123-2136.xlsx to logs_CAK1DAB3-IF-48-03PJJ_20251113-1123-2136.xlsx
Saving logs_CAK1EAB3-IF-48-01PJJ_20251113-1049-1948.xlsx to logs_CAK1EAB3-IF-48-01PJJ_20251113-1049-1948.xlsx
Saving logs_CAK1EAB3-IF-48-02PJJ_20251113-1008-1572.xlsx to logs_CAK1EAB3-IF-48-02PJJ_20251113-1008-1572.xlsx
Saving logs_CAK1EAB3-IF-48-03PJJ_20251113-0921-1421.xlsx to logs_CAK1EAB3-IF-48-03PJJ_20251113-0921-1421.xlsx
Saving MATEMATIKA DISKRIT IF-48-01PJJ [DTO].xlsx to MATEMATIKA D

/tmp/ipython-input-1452670974.py:120: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  logs_df["Time_parsed"] = pd.to_datetime(logs_df["Time"], dayfirst=True, errors='coerce')


🛠️ Final Merge...
Mounted at /content/drive

💾 Menyimpan 4 File Wajib...

✅ DATA ENGINEERING SELESAI & SUKSES!
📂 Folder Output: /content/drive/MyDrive/Semester 7 /Computing project /FINAL_FIX_V7
   (File ke-4 'recommendation_engine.pkl' akan dibuat di tahap ML)


In [ ]:
# ==============================================================================
# CELL 2: MACHINE LEARNING (RETRAIN)
# ==============================================================================
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_score

# 1. LOAD DATA BARU
save_path = "/content/drive/MyDrive/Semester 7 /Computing project /FINAL_FIX_V7"
input_file = os.path.join(save_path, "user_level_features_final_for_ML.csv")

print(f"🔄 Memuat data ML dari: {input_file}")
if not os.path.exists(input_file):
    print("❌ File tidak ditemukan! Jalankan DE dulu.")
else:
    df = pd.read_csv(input_file)
    print(f"✅ Data Loaded: {len(df)} Students")

    # 2. TRAINING
    features = ['mean_score_pct', 'engagement_score', 'consistency_score', 'num_attempts']
    X = df[features].copy()
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
    df['ml_cluster'] = kmeans.fit_predict(X_scaled)

    print(f"📊 Silhouette Score: {silhouette_score(X_scaled, df['ml_cluster']):.4f}")

    # 3. LABELING
    cluster_summary = df.groupby('ml_cluster')[features].mean()
    def get_cluster_label(row):
        score = row['mean_score_pct']
        engage = row['engagement_score']
        if score > 70: return "High Performer (Star)"
        elif engage > 5000:
            if score > 50: return "Active Learner"
            else: return "Hard Worker / Struggling"
        elif engage > 1000: return "Balanced Learner"
        else: return "At Risk / Passive"

    cluster_labels = {}
    print("\n=== CLUSTER INTERPRETATION ===")
    for cluster_id, row in cluster_summary.iterrows():
        label = get_cluster_label(row)
        cluster_labels[cluster_id] = label
        print(f"Cluster {cluster_id} -> {label} (Avg Score: {row['mean_score_pct']:.1f})")

    # 4. SAVE & SYNC
    df['cluster'] = df['ml_cluster']
    df.to_csv(input_file, index=False) # Update CSV User

    model_data = {
        "model": kmeans, "scaler": scaler,
        "cluster_labels": cluster_labels, "features": features
    }
    pkl_path = os.path.join(save_path, "recommendation_engine.pkl")
    with open(pkl_path, "wb") as f:
        pickle.dump(model_data, f)

    print(f"\n✅ ML SELESAI. Model tersimpan di '{pkl_path}'")
    print("\n⬇️ SILAKAN DOWNLOAD 4 FILE INI DARI DRIVE DAN PASANG DI BACKEND:")
    print("   1. merged_score_data_cleaned.csv")
    print("   2. user_level_features_final_for_ML.csv")
    print("   3. merged_logs_data_cleaned.csv")
    print("   4. recommendation_engine.pkl")

🔄 Memuat data ML dari: /content/drive/MyDrive/Semester 7 /Computing project /FINAL_FIX_V7/user_level_features_final_for_ML.csv
✅ Data Loaded: 444 Students
📊 Silhouette Score: 0.7847

=== CLUSTER INTERPRETATION ===
Cluster 0 -> At Risk / Passive (Avg Score: 0.5)
Cluster 1 -> Hard Worker / Struggling (Avg Score: 5.6)
Cluster 2 -> Active Learner (Avg Score: 68.6)
Cluster 3 -> Balanced Learner (Avg Score: 59.2)

✅ ML SELESAI. Model tersimpan di '/content/drive/MyDrive/Semester 7 /Computing project /FINAL_FIX_V7/recommendation_engine.pkl'

⬇️ SILAKAN DOWNLOAD 4 FILE INI DARI DRIVE DAN PASANG DI BACKEND:
   1. merged_score_data_cleaned.csv
   2. user_level_features_final_for_ML.csv
   3. merged_logs_data_cleaned.csv
   4. recommendation_engine.pkl
